# Reset Tables Notebook

Drops all dvdrental Bronze, Silver, and Gold Delta tables and clears streaming checkpoints.
Run this before re-loading from scratch or when switching source data.

**Set `DRY_RUN = true` to preview what would be dropped without actually dropping anything.**

In [ ]:
dbutils.widgets.text("CATALOG",  "workspace")
dbutils.widgets.text("DRY_RUN", "false")  # set to 'true' to preview only

CATALOG  = dbutils.widgets.get("CATALOG")
DRY_RUN  = dbutils.widgets.get("DRY_RUN").lower() == "true"

print(f"Catalog : {CATALOG}")
print(f"Dry run : {DRY_RUN}")

In [ ]:
TABLES = [
    # Bronze
    f"{CATALOG}.bronze.film",
    f"{CATALOG}.bronze.rental",
    f"{CATALOG}.bronze.payment",
    # Silver
    f"{CATALOG}.silver.silver_film",
    f"{CATALOG}.silver.silver_rental",
    f"{CATALOG}.silver.silver_payment",
    # Gold
    f"{CATALOG}.gold.gold_film",
    f"{CATALOG}.gold.gold_rental",
    # Monitoring
    f"{CATALOG}.monitoring.schema_drift_log",
]

CHECKPOINTS = [
    f"/Volumes/{CATALOG}/default/mnt/checkpoints/bronze_cdc",
    f"/Volumes/{CATALOG}/default/mnt/checkpoints/rental_silver",
    f"/Volumes/{CATALOG}/default/mnt/checkpoints/film_silver",
    f"/Volumes/{CATALOG}/default/mnt/checkpoints/payment_silver",
]

In [ ]:
dropped = []
skipped = []

for table in TABLES:
    try:
        spark.table(table)  # will raise if table does not exist
        if DRY_RUN:
            print(f"[DRY RUN] Would drop: {table}")
            dropped.append(table)
        else:
            spark.sql(f"DROP TABLE IF EXISTS {table}")
            print(f"Dropped: {table}")
            dropped.append(table)
    except Exception:
        print(f"Skipped (not found): {table}")
        skipped.append(table)

print(f"\nTables dropped: {len(dropped)}  |  Not found: {len(skipped)}")

In [ ]:
cleared_checkpoints = []
missing_checkpoints = []

for path in CHECKPOINTS:
    try:
        dbutils.fs.ls(path)  # will raise if path does not exist
        if DRY_RUN:
            print(f"[DRY RUN] Would clear checkpoint: {path}")
            cleared_checkpoints.append(path)
        else:
            dbutils.fs.rm(path, recurse=True)
            print(f"Cleared checkpoint: {path}")
            cleared_checkpoints.append(path)
    except Exception:
        print(f"Skipped (not found): {path}")
        missing_checkpoints.append(path)

print(f"\nCheckpoints cleared: {len(cleared_checkpoints)}  |  Not found: {len(missing_checkpoints)}")

In [ ]:
import json

result = {
    "dry_run": DRY_RUN,
    "catalog": CATALOG,
    "tables_dropped": dropped,
    "tables_skipped": skipped,
    "checkpoints_cleared": cleared_checkpoints,
    "checkpoints_skipped": missing_checkpoints,
}
print(json.dumps(result, indent=2))
dbutils.notebook.exit(json.dumps(result))